# FireST-GraphNet  •  **V1** AF post-processing visualization + evaluation

This notebook is a **post-processing notebook only** — it does *not* rerun training or inference. It reads the saved **V1 AF (Active Fire) one-step inference outputs** produced by the legacy V1 exporter (under `.../output/inference_exports_af/`) and generates, for every sample discovered under `PATH1`:

1. one publication-quality PNG per sample (Ground Truth / Prediction / Confusion Map) → `PATH1/visualizations/`
2. one per-sample metrics CSV (F1, IoU, Accuracy, precision/recall, TP/TN/FP/FN, …) → `PATH1/sample_metrics.csv`
3. a small summary CSV with mean / median of F1, IoU, Accuracy → `PATH1/sample_metrics_summary.csv`

### V1 vs V2 layout — what's different?

| aspect | V2 (postprocess_af_inference_exports.ipynb) | **V1 (this notebook)** |
| --- | --- | --- |
| filename | `run27__best_model__train__sample_000000__pred.npy` | `sample00_idx00000_pred.npy` |
| path depth from root | `{run}/{ckpt}/{split}/predictions/file` (5 parts) | `{run}/predictions/file` (3 parts) |
| `run` / `ckpt` / `split` source | parsed from filename | parsed from path + `inference_summary.json` |
| threshold | usually present in `inference_summary.json` | absent → default `0.5` |
| sample_idx | `sample_000000` | `sample00_idx00000` (idx wins) |

Every other cell is identical to the V2 post-processor so the CSV schema, metrics, and PNG layout are unchanged.

Notebook cell organisation:

1. environment / imports
2. Google Drive mount
3. path configuration
4. helper utilities
5. recursive file scanning  *(V1-aware)*
6. data loading logic
7. metric computation functions
8. confusion map generation
9. per-sample visualization export
10. CSV export
11. summary statistics
12. optional quick preview

## 1. Environment / imports

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import json
import glob
import math
import re
import sys
import warnings
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

# Optional dependencies -- imported lazily where used so the notebook still
# runs in environments where torch / cv2 / PIL are unavailable.
try:
    import torch  # noqa: F401
    _HAS_TORCH = True
except Exception:
    _HAS_TORCH = False

try:
    from PIL import Image  # noqa: F401
    _HAS_PIL = True
except Exception:
    _HAS_PIL = False

try:
    import cv2  # noqa: F401
    _HAS_CV2 = True
except Exception:
    _HAS_CV2 = False

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kw):
        return x

warnings.filterwarnings("ignore", category=UserWarning)

print("Python            :", sys.version.split()[0])
print("numpy             :", np.__version__)
print("pandas            :", pd.__version__)
print("matplotlib        :", matplotlib.__version__)
print("torch available   :", _HAS_TORCH)
print("PIL available     :", _HAS_PIL)
print("cv2 available     :", _HAS_CV2)

## 2. Google Drive mount

Mounts `/content/drive` when running on Colab. Silently skipped when running locally (e.g. Windows / Linux workstation).

In [ ]:
def mount_drive_if_colab() -> bool:
    """Try to mount Google Drive under /content/drive. No-op outside Colab."""
    try:
        from google.colab import drive  # type: ignore
    except Exception:
        print("[info] not running on Colab -- skipping Drive mount.")
        return False
    drive.mount("/content/drive", force_remount=False)
    print("[info] Google Drive mounted at /content/drive")
    return True

_DRIVE_MOUNTED = mount_drive_if_colab()

## 3. Path configuration

`PATH1` points to the root folder of already-saved inference exports. The notebook tries a list of likely locations (Colab Drive copy, local OneDrive copy) and picks the first that exists. Override `PATH1` manually if your layout is different.

In [ ]:
# ---- candidate roots (first existing one wins) ----
# V1 AF inference exports land under `<PROJECT>/output/inference_exports_af/`
# (or `outputs/` — the user's Drive folder is plural).  Either is accepted.
_CANDIDATE_ROOTS = [
    # Colab / Google Drive copies (plural `outputs/` first, per user setup)
    "/content/drive/MyDrive/FireST-GraphNet/FireST-GraphNet-Clean/outputs/inference_exports_af",
    "/content/drive/MyDrive/FireST-GraphNet/FireST-GraphNet-Clean/output/inference_exports_af",
    "/content/drive/MyDrive/FireST-GraphNet-Clean/output/inference_exports_af",
    "/content/drive/MyDrive/FireST-GraphNet-Clean/outputs/inference_exports_af",
    # Local OneDrive copy (Windows workstation)
    r"C:\Users\22125\OneDrive - UW-Madison\FireST-GraphNet-Clean\output\inference_exports_af",
    # Linux / generic fallback
    "./output/inference_exports_af",
    "./inference_exports_af",
]

def _first_existing(paths):
    for p in paths:
        if p and Path(p).exists():
            return Path(p)
    return None

PATH1 = _first_existing(_CANDIDATE_ROOTS)
if PATH1 is None:
    PATH1 = Path(_CANDIDATE_ROOTS[0])
    print(f"[warn] none of the candidate roots exist -- defaulting PATH1 to {PATH1}")

# ---- derived output locations ----
VIS_DIR     = PATH1 / "visualizations_postprocess"   # keep V1 exporter's own viz/ untouched
METRICS_CSV = PATH1 / "sample_metrics.csv"
SUMMARY_CSV = PATH1 / "sample_metrics_summary.csv"

# ---- threshold ----
# V1 `inference_summary.json` does NOT record a threshold.  Override here if
# you re-binarise at a different cutoff.
DEFAULT_THRESHOLD = 0.5

# ---- visualisation parameters ----
FIG_DPI       = 220
FIG_SIZE      = (6.0, 14.0)
OVERWRITE_VIZ = False

# ---- preview limits ----
PREVIEW_N = 3

VIS_DIR.mkdir(parents=True, exist_ok=True)

print("PATH1         :", PATH1)
print("VIS_DIR       :", VIS_DIR)
print("METRICS_CSV   :", METRICS_CSV)
print("SUMMARY_CSV   :", SUMMARY_CSV)
print("threshold dflt:", DEFAULT_THRESHOLD)


## 4. Helper utilities

In [ ]:
def log(msg: str) -> None:
    print(msg, flush=True)


def safe_div(num: float, den: float) -> float:
    """Numerically stable division with zero protection."""
    den = float(den)
    if den <= 0.0:
        return 0.0
    return float(num) / den


def to_binary(arr: np.ndarray, threshold: float) -> np.ndarray:
    """Convert a float / int / bool array to a {0,1} uint8 array.

    - booleans are taken as-is
    - integers interpreted as already-binary if max ≤ 1, else thresholded at 0.5
    - floats with range roughly in [0,1] are thresholded at `threshold`
    - floats with larger range are min-max normalised before thresholding
    """
    if arr is None:
        raise ValueError("cannot binarise a None array")
    a = np.asarray(arr)
    if a.dtype == bool:
        return a.astype(np.uint8)
    if np.issubdtype(a.dtype, np.integer):
        if a.max() <= 1:
            return (a > 0).astype(np.uint8)
        return ((a.astype(np.float32) / float(a.max() + 1e-9)) >= 0.5).astype(np.uint8)
    # floating point
    a = a.astype(np.float32)
    vmax = float(a.max()) if a.size else 1.0
    if vmax > 1.5:
        # treat as raw logits / scaled; squash via sigmoid for safety
        a = 1.0 / (1.0 + np.exp(-a))
    return (a >= float(threshold)).astype(np.uint8)


def squeeze_to_2d(arr: np.ndarray) -> np.ndarray:
    """Reduce an array with stray singleton / leading dimensions to a 2D (H, W)."""
    a = np.asarray(arr)
    a = np.squeeze(a)
    if a.ndim == 2:
        return a
    if a.ndim == 3:
        # common: (1, H, W) or (H, W, 1) or (C, H, W) with C==1
        if a.shape[0] == 1:
            return a[0]
        if a.shape[-1] == 1:
            return a[..., 0]
        # otherwise: pick first channel and warn
        return a[0]
    if a.ndim == 4:
        # (B, C, H, W) or (H_out, 1, H, W) etc. -- take [0, 0]
        return a[0, 0]
    raise ValueError(f"cannot squeeze array of shape {a.shape} to 2D")

## 5. Recursive file scanning

Handles the naming convention produced by `run_af_inference_exports.py` (predictions at `*__pred.npy`, GT at `*__gt.npy`) and also tolerates a handful of alternative layouts so the notebook remains adaptable to future runs.

In [ ]:
# Supported prediction / ground-truth file extensions.
_PRED_EXTS = (".npy", ".npz", ".pt", ".pth", ".png")
_GT_EXTS   = (".npy", ".npz", ".pt", ".pth", ".png")

# Filename suffixes that strongly indicate the file type.
# NB: `_hist_last` is *not* in either list, so those files are ignored.
_PRED_SUFFIXES = ("__pred", "_pred", ".pred", "-pred", "__prediction", "_prediction")
_GT_SUFFIXES   = ("__gt", "_gt", ".gt", "-gt", "__target", "_target",
                  "__label", "_label", "__mask", "_mask")


def _strip_pred_suffix(stem: str) -> Optional[str]:
    for s in _PRED_SUFFIXES:
        if stem.endswith(s):
            return stem[: -len(s)]
    return None

def _strip_gt_suffix(stem: str) -> Optional[str]:
    for s in _GT_SUFFIXES:
        if stem.endswith(s):
            return stem[: -len(s)]
    return None


def _candidate_gt_paths(pred_path: Path) -> List[Path]:
    """Given a prediction file path, list plausible matching GT file paths."""
    stem = pred_path.stem
    base = _strip_pred_suffix(stem)
    if base is None:
        base = stem
    out: List[Path] = []
    pred_parent = pred_path.parent
    alt_parents = [pred_parent]
    if pred_parent.name.lower() in {"predictions", "preds", "pred"}:
        alt_parents.append(pred_parent.parent / "ground_truth")
        alt_parents.append(pred_parent.parent / "groundtruth")
        alt_parents.append(pred_parent.parent / "gt")
        alt_parents.append(pred_parent.parent / "targets")
        alt_parents.append(pred_parent.parent / "labels")
    for parent in alt_parents:
        for suf in _GT_SUFFIXES:
            for ext in _GT_EXTS:
                cand = parent / f"{base}{suf}{ext}"
                out.append(cand)
        for ext in _GT_EXTS:
            out.append(parent / f"{base}{ext}")
    seen = set(); uniq: List[Path] = []
    for c in out:
        k = str(c).lower()
        if k in seen:
            continue
        seen.add(k); uniq.append(c)
    return uniq


def _iter_candidate_files(root: Path) -> List[Path]:
    out: List[Path] = []
    for ext in _PRED_EXTS:
        out.extend(root.rglob(f"*{ext}"))
    return out


# ── V1-specific metadata cache (one inference_summary.json per run folder) ──
_SUMMARY_CACHE: Dict[Path, Dict[str, Any]] = {}

def _find_run_summary(pred_path: Path, root: Path) -> Tuple[Optional[Path], Optional[Dict[str, Any]]]:
    """Walk up from the prediction file looking for an inference_summary.json.
    Caches the parsed dict per containing directory."""
    for up in (pred_path.parent, pred_path.parent.parent, pred_path.parent.parent.parent):
        if up == root.parent or up == up.parent:
            break
        cand = up / "inference_summary.json"
        if cand.exists():
            if cand not in _SUMMARY_CACHE:
                try:
                    _SUMMARY_CACHE[cand] = json.loads(cand.read_text(encoding="utf-8"))
                except Exception:
                    _SUMMARY_CACHE[cand] = {}
            return up, _SUMMARY_CACHE[cand]
    return None, None


def _parse_run_ckpt_split(pred_path: Path, root: Path) -> Dict[str, Optional[str]]:
    """V1-aware parser.

    Priority order:
      1. V2-style filename `run__ckpt__split__sample_IDX` (kept for safety so this
         notebook also works if a V2-style export sneaks in).
      2. V1 layout: path parts `{run}/predictions/{file}`; `ckpt` and `split`
         come from the sibling `inference_summary.json`.
      3. Fallback: path components if there are enough of them.
    """
    info: Dict[str, Optional[str]] = {
        "run": None, "ckpt": None, "split": None, "sample_idx": None,
    }
    stem = pred_path.stem
    base = _strip_pred_suffix(stem) or stem

    # V2-style filename -------------------------------------------------------
    m = re.match(
        r"^(?P<run>[^_]+)__(?P<ckpt>[^_]+)__(?P<split>[^_]+)__sample_(?P<idx>\d+)$",
        base,
    )
    if m:
        info["run"]        = m.group("run")
        info["ckpt"]       = m.group("ckpt")
        info["split"]      = m.group("split")
        info["sample_idx"] = int(m.group("idx"))
        return info

    # V1 layout ---------------------------------------------------------------
    try:
        rel = pred_path.relative_to(root).parts
    except ValueError:
        rel = pred_path.parts

    # `.../{run}/predictions/{file}` → run = rel[-3]
    if len(rel) >= 3 and rel[-2].lower() in {"predictions", "preds", "pred"}:
        info["run"] = rel[-3]
    elif len(rel) >= 5:
        info["run"]   = rel[-5]
        info["ckpt"]  = rel[-4]
        info["split"] = rel[-3]

    # sample_idx: V1 uses `sampleNN_idxNNNNN_pred` → prefer the `idxNNNNN` token.
    m_idx = re.search(r"idx[_-]?(\d+)", base)
    if m_idx:
        info["sample_idx"] = int(m_idx.group(1))
    else:
        m_idx = re.search(r"sample[_-]?(\d+)", base)
        if m_idx:
            info["sample_idx"] = int(m_idx.group(1))

    # ckpt / split via sibling inference_summary.json -------------------------
    run_dir, summary = _find_run_summary(pred_path, root)
    if summary:
        if not info["split"] and summary.get("split"):
            info["split"] = str(summary["split"])
        if not info["ckpt"]:
            ckpt_val = summary.get("checkpoint") or summary.get("checkpoint_path")
            if ckpt_val:
                info["ckpt"] = Path(str(ckpt_val)).stem    # e.g. "best_model"
        if not info["run"] and summary.get("run_name"):
            info["run"] = str(summary["run_name"])

    # Conservative defaults when summary is missing
    if info["ckpt"] is None:
        info["ckpt"] = "best_model"
    return info


def _guess_threshold_for(pred_path: Path) -> float:
    """V1 inference_summary.json does NOT store a threshold; return default.
    Kept API-compatible with the V2 notebook so downstream code is unchanged."""
    for up in (pred_path.parent, pred_path.parent.parent, pred_path.parent.parent.parent):
        cand = up / "inference_summary.json"
        if cand.exists():
            try:
                meta = json.loads(cand.read_text(encoding="utf-8"))
                for k in ("threshold", "thr", "thr_best", "best_threshold"):
                    if k in meta and meta[k] is not None:
                        return float(meta[k])
            except Exception:
                pass
    return DEFAULT_THRESHOLD


def scan_samples(root: Path) -> List[Dict[str, Any]]:
    """Build a list of {pred, gt, meta} records discovered under `root`."""
    cand = _iter_candidate_files(root)
    cand = [p for p in cand
            if VIS_DIR not in p.parents and p != METRICS_CSV and p != SUMMARY_CSV]
    pred_files = [p for p in cand if _strip_pred_suffix(p.stem) is not None]
    pred_files.sort()

    recs: List[Dict[str, Any]] = []
    for p in pred_files:
        gt_path: Optional[Path] = None
        for g in _candidate_gt_paths(p):
            if g.exists():
                gt_path = g
                break
        info = _parse_run_ckpt_split(p, root)
        thr  = _guess_threshold_for(p)
        recs.append({
            "pred_path": p,
            "gt_path": gt_path,
            "threshold": thr,
            **info,
        })
    return recs


samples = scan_samples(PATH1)
n_with_gt = sum(1 for r in samples if r["gt_path"] is not None)
log(f"Discovered {len(samples)} prediction file(s); {n_with_gt} have a matching GT.")
if samples:
    log("First 3 records:")
    for r in samples[:3]:
        log(f"  pred={r['pred_path'].name}  gt={r['gt_path'].name if r['gt_path'] else '-'}  "
            f"run={r['run']} ckpt={r['ckpt']} split={r['split']} idx={r['sample_idx']} thr={r['threshold']}")


## 6. Data loading logic

Modular loaders for `.npy` / `.npz` / `.pt` / `.pth` / `.png`. Each returns a 2-D numpy array.

In [ ]:
def _load_npy(path: Path) -> np.ndarray:
    return np.load(str(path), allow_pickle=False)


def _load_npz(path: Path) -> np.ndarray:
    with np.load(str(path), allow_pickle=False) as z:
        # pick the first array whose name looks like pred / gt / arr_0
        keys = list(z.keys())
        preferred = [k for k in keys if re.search(r"(pred|prob|logit|mask|gt|label|target|arr_0)", k, re.I)]
        k = preferred[0] if preferred else keys[0]
        return np.array(z[k])


def _load_pt(path: Path) -> np.ndarray:
    if not _HAS_TORCH:
        raise RuntimeError("torch is required to load .pt/.pth files")
    obj = torch.load(str(path), map_location="cpu", weights_only=False)
    if isinstance(obj, dict):
        for k in ("prediction", "pred", "prob", "logits", "mask", "gt", "target", "label"):
            if k in obj:
                obj = obj[k]
                break
        else:
            # single-entry dict -> take it; else bail
            if len(obj) == 1:
                obj = next(iter(obj.values()))
            else:
                raise ValueError(f"ambiguous dict keys in {path.name}: {list(obj.keys())[:6]}")
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().numpy()
    return np.asarray(obj)


def _load_png(path: Path) -> np.ndarray:
    if _HAS_PIL:
        img = np.array(Image.open(str(path)).convert("L"))
        return img.astype(np.float32) / 255.0
    if _HAS_CV2:
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise IOError(f"cannot read image {path}")
        return img.astype(np.float32) / 255.0
    raise RuntimeError("neither PIL nor cv2 are available to read PNGs")


_LOADERS = {
    ".npy": _load_npy,
    ".npz": _load_npz,
    ".pt":  _load_pt,
    ".pth": _load_pt,
    ".png": _load_png,
}


def load_array(path: Path) -> np.ndarray:
    """Dispatch to the right loader based on extension; return a 2-D numpy array."""
    ext = path.suffix.lower()
    if ext not in _LOADERS:
        raise ValueError(f"unsupported file extension: {path}")
    arr = _LOADERS[ext](path)
    arr = squeeze_to_2d(arr)
    return np.asarray(arr)

## 7. Metric computation

Per-sample TP/TN/FP/FN, precision, recall, F1, IoU, accuracy. All divisions are zero-guarded.

In [ ]:
def confusion_counts(pred_bin: np.ndarray, gt_bin: np.ndarray) -> Dict[str, int]:
    """Return {tp, tn, fp, fn, total} for two {0,1} uint8 arrays."""
    assert pred_bin.shape == gt_bin.shape, (
        f"shape mismatch: pred={pred_bin.shape} gt={gt_bin.shape}"
    )
    p = pred_bin.astype(bool)
    g = gt_bin.astype(bool)
    tp = int(np.logical_and(p,  g).sum())
    tn = int(np.logical_and(~p, ~g).sum())
    fp = int(np.logical_and(p,  ~g).sum())
    fn = int(np.logical_and(~p, g).sum())
    total = int(p.size)
    return {"tp": tp, "tn": tn, "fp": fp, "fn": fn, "total": total}


def segmentation_metrics(pred_bin: np.ndarray, gt_bin: np.ndarray) -> Dict[str, float]:
    """Compute standard segmentation metrics, safe against empty masks."""
    c = confusion_counts(pred_bin, gt_bin)
    tp, tn, fp, fn, total = c["tp"], c["tn"], c["fp"], c["fn"], c["total"]
    precision = safe_div(tp, tp + fp)
    recall    = safe_div(tp, tp + fn)
    f1        = safe_div(2.0 * precision * recall, precision + recall)
    iou       = safe_div(tp, tp + fp + fn)
    accuracy  = safe_div(tp + tn, total)
    return {
        **c,
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
        "iou":       iou,
        "accuracy":  accuracy,
    }

## 8. Confusion-map generation

Builds an (H, W) integer map with the colour encoding:

- 0 → TN (black)
- 1 → TP (red)
- 2 → FP (yellow)
- 3 → FN (blue)

In [ ]:
# Canonical confusion colour encoding: TN black, TP red, FP yellow, FN blue.
_CONF_CODE_TN = 0
_CONF_CODE_TP = 1
_CONF_CODE_FP = 2
_CONF_CODE_FN = 3
_CONF_COLORS  = ["#000000", "#d62728", "#ffd400", "#1f77b4"]   # TN, TP, FP, FN
_CONF_CMAP    = ListedColormap(_CONF_COLORS, name="conf_map")
_CONF_LABELS  = {
    _CONF_CODE_TP: "TP (Red)",
    _CONF_CODE_TN: "TN (Black)",
    _CONF_CODE_FP: "FP (Yellow)",
    _CONF_CODE_FN: "FN (Blue)",
}


def confusion_map(pred_bin: np.ndarray, gt_bin: np.ndarray) -> np.ndarray:
    """Pixel-wise confusion integer map, shape (H, W), dtype uint8."""
    assert pred_bin.shape == gt_bin.shape
    p = pred_bin.astype(bool)
    g = gt_bin.astype(bool)
    m = np.full(p.shape, _CONF_CODE_TN, dtype=np.uint8)
    m[np.logical_and(p, g)]   = _CONF_CODE_TP
    m[np.logical_and(p, ~g)]  = _CONF_CODE_FP
    m[np.logical_and(~p, g)]  = _CONF_CODE_FN
    return m

## 9. Per-sample visualization export

One figure with **3 rows**: Ground Truth, Prediction, Confusion Map. Saved as PNG under `PATH1/visualizations/`. Deterministic filename: `{sample_id}.png` (with a prefix when needed to avoid collisions across runs).

In [ ]:
# Discrete black/white colourmap for binary GT / prediction panels.
_BIN_CMAP = ListedColormap(["#000000", "#ffffff"], name="bw")


def build_sample_id(rec: Dict[str, Any]) -> str:
    """Deterministic identifier used for both the CSV `sample_id` column and
    the output PNG filename. Falls back to a sequential index when needed."""
    run   = rec.get("run")
    ckpt  = rec.get("ckpt")
    split = rec.get("split")
    idx   = rec.get("sample_idx")
    if run and ckpt and split and idx is not None:
        return f"{run}__{ckpt}__{split}__sample_{int(idx):06d}"
    # Otherwise use the original stem (stripping any __pred suffix).
    stem = rec["pred_path"].stem
    base = _strip_pred_suffix(stem) or stem
    return base


def save_sample_visualisation(
    save_path: Path,
    sample_id: str,
    gt_bin: np.ndarray,
    pred_bin: np.ndarray,
    conf: np.ndarray,
    metrics: Dict[str, float],
    threshold: float,
) -> None:
    """Render and save the 3-row figure (GT / Prediction / Confusion)."""
    fig, axes = plt.subplots(
        3, 1,
        figsize=FIG_SIZE,
        facecolor="white",
        gridspec_kw={"hspace": 0.18},
    )

    # Row 1 -- Ground Truth
    ax = axes[0]
    ax.imshow(gt_bin, cmap=_BIN_CMAP, vmin=0, vmax=1, interpolation="nearest")
    ax.set_title("Ground Truth", fontsize=12)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_edgecolor("#333"); s.set_linewidth(0.8)

    # Row 2 -- Prediction
    ax = axes[1]
    ax.imshow(pred_bin, cmap=_BIN_CMAP, vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(f"Prediction (thr={threshold:g})", fontsize=12)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_edgecolor("#333"); s.set_linewidth(0.8)

    # Row 3 -- Confusion Map with legend
    ax = axes[2]
    ax.imshow(conf, cmap=_CONF_CMAP, vmin=0, vmax=3, interpolation="nearest")
    ax.set_title("Confusion Map", fontsize=12)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_edgecolor("#333"); s.set_linewidth(0.8)
    legend_handles = [
        Patch(facecolor=_CONF_COLORS[_CONF_CODE_TP], edgecolor="#333", label="TP (Red)"),
        Patch(facecolor=_CONF_COLORS[_CONF_CODE_TN], edgecolor="#333", label="TN (Black)"),
        Patch(facecolor=_CONF_COLORS[_CONF_CODE_FP], edgecolor="#333", label="FP (Yellow)"),
        Patch(facecolor=_CONF_COLORS[_CONF_CODE_FN], edgecolor="#333", label="FN (Blue)"),
    ]
    ax.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.22),
        ncol=4,
        frameon=False,
        fontsize=9,
    )

    # Title + footer with metrics
    fig.suptitle(f"Sample: {sample_id}", fontsize=13, y=0.995)
    footer = (
        f"F1={metrics['f1']:.3f}   IoU={metrics['iou']:.3f}   Acc={metrics['accuracy']:.3f}"
        f"   Precision={metrics['precision']:.3f}   Recall={metrics['recall']:.3f}"
    )
    fig.text(0.5, 0.01, footer, ha="center", va="bottom", fontsize=10)

    fig.tight_layout(rect=(0, 0.03, 1, 0.97))
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(str(save_path), dpi=FIG_DPI, bbox_inches="tight", facecolor="white")
    plt.close(fig)

## 10. Batch processing — metrics + visualizations

Iterates over every discovered sample, loads prediction + GT, binarises, computes metrics, writes the PNG, and collects metrics rows. Problems with individual samples are logged as warnings rather than aborting the run.

In [ ]:
def process_one_sample(rec: Dict[str, Any], seq_idx: int) -> Optional[Dict[str, Any]]:
    pred_path: Path = rec["pred_path"]
    gt_path:   Optional[Path] = rec["gt_path"]
    threshold: float = float(rec.get("threshold") or DEFAULT_THRESHOLD)

    sample_id = build_sample_id(rec)

    row: Dict[str, Any] = {
        "sample_index": seq_idx,
        "sample_id":    sample_id,
        "event_id":     "",  # not persisted in the current export format
        "run":          rec.get("run"),
        "ckpt":         rec.get("ckpt"),
        "split":        rec.get("split"),
        "prediction_source_file": str(pred_path),
        "gt_source_file":         str(gt_path) if gt_path is not None else "",
        "threshold":   threshold,
        "status":      "pending",
        "message":     "",
    }

    if gt_path is None:
        row["status"]  = "skipped"
        row["message"] = "no matching GT found"
        log(f"[warn] {sample_id}: no matching GT found for {pred_path.name}")
        return row

    try:
        pred_arr = load_array(pred_path)
        gt_arr   = load_array(gt_path)
    except Exception as e:
        row["status"]  = "error"
        row["message"] = f"load failed: {e!r}"
        log(f"[warn] {sample_id}: load failed -- {e}")
        return row

    if pred_arr.shape != gt_arr.shape:
        row["status"]  = "error"
        row["message"] = f"shape mismatch pred={pred_arr.shape} gt={gt_arr.shape}"
        log(f"[warn] {sample_id}: shape mismatch pred={pred_arr.shape} gt={gt_arr.shape}")
        return row

    pred_bin = to_binary(pred_arr, threshold=threshold)
    gt_bin   = to_binary(gt_arr,   threshold=0.5)

    metrics = segmentation_metrics(pred_bin, gt_bin)
    conf    = confusion_map(pred_bin, gt_bin)

    row.update({
        "tp":        metrics["tp"],
        "tn":        metrics["tn"],
        "fp":        metrics["fp"],
        "fn":        metrics["fn"],
        "precision": metrics["precision"],
        "recall":    metrics["recall"],
        "f1":        metrics["f1"],
        "iou":       metrics["iou"],
        "accuracy":  metrics["accuracy"],
        "gt_positive_pixels":   int(gt_bin.sum()),
        "pred_positive_pixels": int(pred_bin.sum()),
        "total_pixels":         int(gt_bin.size),
    })

    vis_path = VIS_DIR / f"{sample_id}.png"
    row["visualization_file"] = str(vis_path)
    if (not vis_path.exists()) or OVERWRITE_VIZ:
        try:
            save_sample_visualisation(
                vis_path, sample_id, gt_bin, pred_bin, conf, metrics, threshold,
            )
        except Exception as e:
            row["status"]  = "error"
            row["message"] = f"viz failed: {e!r}"
            log(f"[warn] {sample_id}: viz failed -- {e}")
            return row

    row["status"] = "ok"
    return row


log(f"Processing {len(samples)} sample(s) -> {VIS_DIR}")
rows: List[Dict[str, Any]] = []
for i, rec in enumerate(tqdm(samples, desc="post-processing", unit="sample")):
    r = process_one_sample(rec, seq_idx=i)
    if r is not None:
        rows.append(r)

log(f"Done -- {sum(1 for r in rows if r['status'] == 'ok')}/{len(rows)} samples processed OK.")

## 11. CSV export

Writes `PATH1/sample_metrics.csv` (one row per sample) with a stable column order.

In [ ]:
_METRICS_COLUMNS = [
    "sample_index", "sample_id", "event_id", "run", "ckpt", "split",
    "prediction_source_file", "gt_source_file", "visualization_file",
    "threshold",
    "tp", "tn", "fp", "fn",
    "precision", "recall", "f1", "iou", "accuracy",
    "gt_positive_pixels", "pred_positive_pixels", "total_pixels",
    "status", "message",
]

df = pd.DataFrame(rows)
for col in _METRICS_COLUMNS:
    if col not in df.columns:
        df[col] = ""
df = df[_METRICS_COLUMNS]
df.to_csv(str(METRICS_CSV), index=False)

log(f"Wrote per-sample metrics -> {METRICS_CSV}   (rows={len(df)})")
df.head(8)

## 12. Summary statistics

Aggregates mean and median of F1 / IoU / Accuracy across all samples whose status is `ok`. Prints the numbers and also saves them to `sample_metrics_summary.csv`.

In [ ]:
_ok = df[df["status"] == "ok"].copy()

if len(_ok) == 0:
    log("[warn] no samples with status=ok -- summary skipped.")
    summary_rows: List[Dict[str, Any]] = []
else:
    def _stat(col: str, fn) -> float:
        vals = pd.to_numeric(_ok[col], errors="coerce").dropna().to_numpy()
        return float(fn(vals)) if vals.size else float("nan")

    summary_global = {
        "scope":       "ALL",
        "n_samples":   int(len(_ok)),
        "mean_f1":     _stat("f1",       np.mean),
        "mean_iou":    _stat("iou",      np.mean),
        "mean_acc":    _stat("accuracy", np.mean),
        "median_f1":   _stat("f1",       np.median),
        "median_iou":  _stat("iou",      np.median),
        "median_acc":  _stat("accuracy", np.median),
        "mean_precision": _stat("precision", np.mean),
        "mean_recall":    _stat("recall",    np.mean),
    }
    log("Global summary (status=ok only):")
    for k, v in summary_global.items():
        log(f"  {k:<16} : {v}")

    # Per-(run, ckpt, split) breakdown when those columns are populated.
    summary_rows = [summary_global]
    group_cols = [c for c in ("run", "ckpt", "split") if c in _ok.columns]
    if group_cols and _ok[group_cols].notna().any().all():
        for keys, sub in _ok.groupby(group_cols):
            if not isinstance(keys, tuple):
                keys = (keys,)
            scope = "/".join(str(k) for k in keys)
            summary_rows.append({
                "scope":       scope,
                "n_samples":   int(len(sub)),
                "mean_f1":     float(pd.to_numeric(sub["f1"],       errors="coerce").mean()),
                "mean_iou":    float(pd.to_numeric(sub["iou"],      errors="coerce").mean()),
                "mean_acc":    float(pd.to_numeric(sub["accuracy"], errors="coerce").mean()),
                "median_f1":   float(pd.to_numeric(sub["f1"],       errors="coerce").median()),
                "median_iou":  float(pd.to_numeric(sub["iou"],      errors="coerce").median()),
                "median_acc":  float(pd.to_numeric(sub["accuracy"], errors="coerce").median()),
                "mean_precision": float(pd.to_numeric(sub["precision"], errors="coerce").mean()),
                "mean_recall":    float(pd.to_numeric(sub["recall"],    errors="coerce").mean()),
            })

summary_df = pd.DataFrame(summary_rows)
if len(summary_df):
    summary_df.to_csv(str(SUMMARY_CSV), index=False)
    log(f"Wrote summary -> {SUMMARY_CSV}   (scopes={len(summary_df)})")
summary_df

## 13. Quick preview (optional)

Displays the first `PREVIEW_N` generated PNGs inline so you can sanity-check the visualisations without leaving the notebook.

In [ ]:
preview_files = sorted(VIS_DIR.glob("*.png"))[:PREVIEW_N]
if not preview_files:
    log("[info] no PNGs to preview.")
else:
    from IPython.display import Image as IPyImage, display
    for p in preview_files:
        log(f"– {p.name}")
        display(IPyImage(filename=str(p)))

---
### Output locations

| artefact | path |
| --- | --- |
| per-sample PNGs | `PATH1/visualizations/*.png` |
| per-sample metrics | `PATH1/sample_metrics.csv` |
| summary statistics | `PATH1/sample_metrics_summary.csv` |

This notebook does **not** rerun training or inference — it only post-processes artefacts already exported by `run_af_inference_exports.py`.